In [3]:
# --- Sanity probe for sharded pipeline components (NO API CALLS) -------------
from pathlib import Path
import json
import pandas as pd

# Import the exact components your run_experiment_sharded uses
from humaidclf.io import load_tsv, plan_run_dirs
from humaidclf.runner import _present_labels_from_df
from humaidclf.runner_sharded import slice_rules_for_labels
from humaidclf.batch import build_requests_jsonl_S
from humaidclf.stratify import stratified_k_shards

# Your compact rules (unchanged)
from rules import RULES_1

# ====== CONFIG ======
DATASET_PATH = "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv"
MODEL        = "gpt-4o-mini"
TAG          = "probe-sharded-dryrun"
OUT_ROOT     = "runs_probe"

# 1) Load dataset the same way
df_full = load_tsv(DATASET_PATH).reset_index(drop=False).rename(columns={"index": "_order"})
df_full["tweet_id"] = df_full["tweet_id"].astype(str)
print(f"[OK] Loaded dataset: {DATASET_PATH}  (rows={len(df_full)})")

# 2) Truth-scoped labels (uses the same helper as runner_sharded)
truth_labels = _present_labels_from_df(df_full)
print("[OK] Truth-scoped labels:", truth_labels)

# 3) Rules: slice to those labels using the function from runner_sharded
rules_scoped = slice_rules_for_labels(RULES_1, truth_labels)
print("\n--- Scoped rules (exact lines) ---")
print(rules_scoped or "[EMPTY]")
print("--- End scoped rules ---\n")

# Quick guardrails
assert truth_labels, "No truth labels detected."
assert rules_scoped.strip(), "Scoped rules are EMPTY — check RULES_1 format or label names."

# 4) Plan directories like the sharded runner does
plan_root = plan_run_dirs(DATASET_PATH, out_root=OUT_ROOT, model=MODEL, tag=TAG)
run_dir   = Path(plan_root["dir"])
(run_dir / "shards").mkdir(parents=True, exist_ok=True)
# Persist the rules you’ll actually use, for eyeballing
(run_dir / "scoped_rules.txt").write_text(rules_scoped, encoding="utf-8")
print("[OK] Wrote scoped_rules.txt →", run_dir / "scoped_rules.txt")

# 5) Build a tiny shard and generate requests.jsonl (NO submission)
#    Use stratified_k_shards so the code path matches the real one.
shards = stratified_k_shards(df_full, label_col="class_label", k=2, seed=42)
df_shard = shards[0].copy()
df_shard["tweet_id"] = df_shard["tweet_id"].astype(str)

shard_dir = run_dir / "shards" / "shard01"
shard_dir.mkdir(parents=True, exist_ok=True)
req_path = shard_dir / "requests.jsonl"

# Keep this shard very small for quick inspection
df_small = df_shard.head(5).copy()
df_small.to_csv(shard_dir / "shard01.tsv", sep="\t", index=False)

build_requests_jsonl_S(
    df_small,
    out_path=str(req_path),
    rules=rules_scoped,               # <-- the filtered rules
    model=MODEL,
    temperature=0.0,
    labels_override=truth_labels,     # <-- event-level label set
    allow_single_label_bypass=False,  # <-- exactly like sharded mode
)
print("[OK] Built requests.jsonl →", req_path)

# 6) Peek at the first request to verify messages/rules/schema
with open(req_path, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()

obj = json.loads(first_line)
print("\n--- First JSONL record (pretty) ---")
print(json.dumps(obj, indent=2)[:2000])  # truncated preview

# Optional validations:
# - rules text really present in user/system content
msg_blocks = obj.get("body", {}).get("messages", [])
joined_msgs = "\n\n".join([m.get("content", "") if isinstance(m.get("content", ""), str)
                           else str(m.get("content")) for m in msg_blocks])
assert any("- " in rules_scoped for _ in [0]) and (rules_scoped.splitlines()[0] in joined_msgs), \
    "Scoped rules do not appear inside the request messages."

# - enum is the truth_labels
resp_fmt = obj.get("body", {}).get("response_format")
if not resp_fmt:
    # Some builders embed schema elsewhere; try typical location used in your code:
    pass
print("\n[OK] Sanity probe finished. Inspect the printed labels/rules and the JSONL preview above.")


[OK] Loaded dataset: Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv  (rows=435)
[OK] Truth-scoped labels: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support', 'other_relevant_information', 'not_humanitarian']

--- Scoped rules (exact lines) ---
- caution_and_advice: warnings/instructions/tips
- displaced_people_and_evacuations: evacuations, relocation, shelters
- infrastructure_and_utility_damage: damage/outages to roads/bridges/power/water/buildings
- injured_or_dead_people: injuries, casualties, fatalities
- requests_or_urgent_needs: asking for help/supplies/SOS
- rescue_volunteering_or_donation_effort: offering help, donation, organizing aid
- sympathy_and_support: prayers/condolences, no actionable info
- other_relevant_information: on-topic but none of the above
- not_humanitarian: unrelated to